# Model Track A — KNN & Logistic Regression

Pipeline của P2 (Dương): đánh giá xác suất đa lớp bằng nested 10-fold CV, với preprocessing được fit riêng trong từng fold để tránh leakage. Tập `aio26_val.csv` không được trộn vào điểm CV chính.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_track_a import run_experiment


## 1. Nested cross-validation

Outer 10-fold dùng `data/interim/fold_id.csv`. Trong mỗi outer training fold, inner 3-fold chọn `k ∈ {3,5,11,21,31}` cho KNN và `C ∈ {0.1,1,10}` cho Logistic Regression theo log loss.

In [2]:
outputs = run_experiment()
scores = outputs['scores']
tuning = outputs['tuning']
ablation = outputs['ablation']
ablation_summary = outputs['ablation_summary']

assert list(scores.columns) == ['model', 'fold', 'log_loss', 'accuracy', 'macro_f1']
assert len(scores) == 20
assert not scores.duplicated(['model', 'fold']).any()
scores


,model,fold,log_loss,accuracy,macro_f1
0,KNN,0,0.713587,0.808333,0.509942
1,Logistic_Regression,0,0.453988,0.826042,0.558370
2,KNN,1,0.850400,0.829167,0.556255
3,Logistic_Regression,1,0.438816,0.832292,0.539371
4,KNN,2,0.782371,0.821875,0.525655
5,Logistic_Regression,2,0.437692,0.833333,0.564527
6,KNN,3,0.821805,0.848958,0.552231
7,Logistic_Regression,3,0.410677,0.857292,0.588143
8,KNN,4,0.677277,0.829167,0.529745
9,Logistic_Regression,4,0.426290,0.839583,0.605519


In [3]:
metric_cols = ['log_loss', 'accuracy', 'macro_f1']
score_summary = scores.groupby('model')[metric_cols].agg(['mean', 'std'])
score_summary


log_loss            accuracy            macro_f1  \
                         mean       std      mean       std      mean   
model                                                                   
KNN                  0.779374  0.096318  0.821458  0.012405  0.525700   
Logistic_Regression  0.439808  0.022817  0.833854  0.011059  0.560388   

                               
                          std  
model                          
KNN                  0.016564  
Logistic_Regression  0.023766

In [4]:
pd.crosstab(tuning['model'], tuning['best_params'])


best_params,"{""model__C"": 1.0}","{""model__n_neighbors"": 31}"
model,,
KNN,0,10
Logistic_Regression,10,0


## 2. Ablation scaling cho KNN

Hai biến thể đều dùng `k=5`, cùng outer folds và cùng feature engineering. Khác biệt duy nhất là bật/tắt `RobustScaler`, nhờ đó phép so sánh cô lập ảnh hưởng của scaling.

In [5]:
ablation_summary


,model,log_loss_mean,log_loss_std,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,zero_probability_rate_mean,zero_probability_rate_std,mean_max_probability_mean,mean_max_probability_std,log_loss_delta_vs_scaled
0,KNN_scaled,2.453543,0.159180,0.817604,0.009912,0.562270,0.016717,0.494444,0.004814,0.863000,0.003572,0.000000
1,KNN_unscaled,3.076369,0.247356,0.775625,0.009410,0.519751,0.014403,0.456181,0.008411,0.831625,0.006822,0.622826


In [6]:
scaled = ablation_summary.set_index('model').loc['KNN_scaled']
unscaled = ablation_summary.set_index('model').loc['KNN_unscaled']
delta = unscaled['log_loss_mean'] - scaled['log_loss_mean']
print(f'Scaling làm log loss KNN giảm {delta:.4f}.')
print(f"KNN scaled vẫn có {scaled['zero_probability_rate_mean']:.1%} phần tử xác suất bằng 0.")
print('Vì log loss phạt rất mạnh xác suất 0 ở lớp đúng, log loss KNN có thể cao dù accuracy/F1 hợp lý; đây là đặc tính xác suất thô của KNN, không phải lỗi tính metric.')


Scaling làm log loss KNN giảm 0.6228.
KNN scaled vẫn có 49.4% phần tử xác suất bằng 0.
Vì log loss phạt rất mạnh xác suất 0 ở lớp đúng, log loss KNN có thể cao dù accuracy/F1 hợp lý; đây là đặc tính xác suất thô của KNN, không phải lỗi tính metric.


## 3. Artifact

Notebook ghi `scores_track_a.csv`, `tuning_track_a.csv`, `ablation_scaling_track_a.csv` vào `results/` và bảng ablation tổng hợp vào `tables/`. File điểm chính giữ đúng schema chung để P4 dùng kiểm định ghép cặp.